# Models exploration

## Hierachical approach
- Since we know the taxonomy, a 2 level hierachical approach can be applied
- Start from training models on clean audio sound

### 1. Top-down approach (hard hierarchical combination)
- Model 1 : predict family
- A. Model 1...n_families : predict species conditioned by the family OR B. Single global model with masking.

#### Inference. Hard combination
During inference, predict family, then predict species.

**Problem** : wrong family -> wrong species (error propagation)

#### Inference. Soft combination
If model outputs probabilities, then
$$
P(y|x) = P(family_y | x) \cdot P(y | x, family_y)
$$

Also, instead of training n_families models, one model can be trained to predict species, then (with hypothesis that model implicitely learned hierachy),

$$
P(y|x, family_y) = P(y|x)
$$

## Train/test separation
Simple stratification per labels

## Features
#### Features extraction 
0. Waveform [experimental purposes only, baseline]
1. MFCC (with delta, delta-delta)
2. Spectral features : useful since different species often occupy different frequency ranges.
- Spectral centroid -> "brightness"
- Spectral bandwidth -> spread of frequencies
- Spectral rolloff -> frequency below which X% energy lies
- Spectral contrast -> peaks vs valleys

3. RMS energy, zero-crossing rate (distinguish between tonal calls vs noisy env)

#### Pooling/aggregation
Since every audio is of different duration, extracted features must be aggregated.

##### Global pooling : 1 audio = 1 unit, aggregate over all frames
- Mean, std, max, min, percentiles, skewness & kurtosis

##### Chunking : 1 audio = N units/chunks, aggregate over chunks
- Split every audio into N chunks
- Aggregate per segment
- Concatenate aggregates features

## Questions
1. Training on clean audio, then how to adapt models to soundscape
2. Some classes are present only in soundscape, how to adapt models and detect these classes

In [17]:
import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

import librosa
import librosa.display
import soundfile as sf

from IPython.display import Markdown

from birdclef_2026_ml.notebook_utils import init_notebook
from birdclef_2026_ml.paths import PATHS
from birdclef_2026_ml.audio_utils import load_train_audio
from birdclef_2026_ml.eda import *
from birdclef_2026_ml.preprocess import *
from birdclef_2026_ml.constants import SAMPLE_RATE

from birdclef_2026_ml.data_split import split_audio_train_val
from birdclef_2026_ml.feature_engineering import *

init_notebook()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
train = pd.read_parquet(PATHS["proc_train"])
#soundscapes = pd.read_csv(PATHS["raw_soundscape_labels"])

In [16]:
train

,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection,duration
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,NaN,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat,18.024000
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,NaN,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat,28.392000
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,NaN,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat,80.016000
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,NaN,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat,61.272000
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,NaN,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat,71.064000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31986,yeofly1,[],[call],-6.7281,-76.4181,Tolmomyias sulphurescens,Yellow-olive Flatbill,Aves,16567,GABRIEL LEITE,by-nc-sa,4.0,https://xeno-canto.org/762473/download,yeofly1/XC762473.ogg,XC,21.209937
31987,yeofly1,[],"[call, song]",-16.0538,-49.6040,Tolmomyias sulphurescens,Yellow-olive Flatbill,Aves,16567,JAYRSON ARAUJO DE OLIVEIRA,by-nc-sa,5.0,https://xeno-canto.org/818328/download,yeofly1/XC818328.ogg,XC,13.479188
31988,yeofly1,[],[song],5.9002,-74.8485,Tolmomyias sulphurescens,Yellow-olive Flatbill,Aves,16567,Jerome Fischer,by-nc-sa,4.0,https://xeno-canto.org/425545/download,yeofly1/XC425545.ogg,XC,80.039187
31989,yeofly1,[],[song],3.3821,-61.4464,Tolmomyias sulphurescens,Yellow-olive Flatbill,Aves,16567,GABRIEL LEITE,by-nc-sa,4.0,https://xeno-canto.org/456230/download,yeofly1/XC456230.ogg,XC,11.376000


In [11]:
feat_cfg = FeatureConfig(include_waveform_stats=True,
                         include_mfcc=False,
                         include_delta=False, 
                         include_delta2=False,
                         include_energy=False,
                         include_spectral=False)
pool_cfg = PoolingConfig()
#X, features = extract_features_from_row(df=train, idx=0, feature_cfg=feat_cfg, pooling_mode="global", pooling_cfg=pool_cfg)

In [12]:
X, features = build_feature_matrix_from_df(train.head(100),
                                           feat_cfg,
                                           "global", 
                                           pool_cfg)

In [13]:
X 

array([[-2.32420518e-05,  1.44229670e-02, -2.22511023e-01,
         1.78944245e-01, -1.60916165e-02, -4.24676364e-06,
         1.60828952e-02],
       [-1.78458995e-05,  8.56639941e-03, -1.04393303e-01,
         8.74324068e-02, -9.92877120e-03, -2.71562958e-05,
         9.92312282e-03],
       [-2.61735322e-05,  9.88606617e-03, -2.35330895e-01,
         1.95811078e-01, -1.19998755e-02, -3.76734351e-05,
         1.19869289e-02],
       [-2.71499010e-05,  7.49626158e-02, -4.10867810e-01,
         4.07455802e-01, -9.39529076e-02, -2.58086175e-05,
         9.38357420e-02],
       [-1.56046917e-05,  4.00450785e-02, -1.98079810e-01,
         2.03110278e-01, -5.10765370e-02, -1.93573192e-06,
         5.10815889e-02],
       [-1.42961520e-07,  1.94684117e-02, -2.25910634e-01,
         2.22099975e-01, -2.34350111e-02,  9.67479874e-09,
         2.34149992e-02],
       [-2.23268284e-06,  4.91567491e-02, -2.36237228e-01,
         2.30589092e-01, -5.91481715e-02,  1.12549878e-05,
         5.9698929